In [29]:
print("hello world")

hello world


PM Task
Load the two datasets
Clean the them (thoroughly)
Measure at least 1 data engineering metric in the data cleaning process (ie dropped rows)
MVP: Output the cleaned files as a local csv.
Stretch:

Output the cleaned files into the local SSMS (use SQLAlchemy); create a new DB for this.
Refactor your code into modular functions.

In [30]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



Read in both original files

In [31]:
import pandas as pd

books = pd.read_csv("data/03_Library Systembook.csv")
customers = pd.read_csv("data/03_Library SystemCustomers.csv")

# print("Books:")
# display(books.head())

# print("Customers:")
# display(customers.head())

In [32]:
books_raw = books.copy()

print("Shape:", books_raw.shape)
display(books_raw.head())

print("\nColumns:")
print(books_raw.columns.tolist())

print("\nData types:")
print(books_raw.dtypes)

print("\nMissing Values:")
display(books_raw.isna().sum().to_frame("missing_count"))

print("\nDuplicate Rows:")
print(books_raw.duplicated().sum())


Shape: (114, 6)


,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1.0,Catcher in the Rye,"""20/02/2023""",25/02/2023,2 weeks,1.0
1,2.0,Lord of the rings the two towers,"""24/03/2023""",21/03/2023,2 weeks,2.0
2,3.0,Lord of the rings the return of the kind,"""29/03/2023""",25/03/2023,2 weeks,3.0
3,4.0,The hobbit,"""02/04/2023""",25/03/2023,2 weeks,4.0
4,5.0,Dune,"""02/04/2023""",25/03/2023,2 weeks,5.0



Columns:
['Id', 'Books', 'Book checkout', 'Book Returned', 'Days allowed to borrow', 'Customer ID']

Data types:
Id                        float64
Books                      object
Book checkout              object
Book Returned              object
Days allowed to borrow     object
Customer ID               float64
dtype: object

Missing Values:


,missing_count
Id,93
Books,94
Book checkout,93
Book Returned,93
Days allowed to borrow,93
Customer ID,94



Duplicate Rows:
92


In [33]:
books_clean = books_raw.copy()

rows_before = len(books_clean)

#Delete the blank rows
books_clean = books_clean.dropna(how="all")

rows_after = len(books_clean)

print("Rows Before:", rows_before)
print("Rows After:", rows_after)
print("Blank rows Removed:", rows_before - rows_after)

display(books_clean)

Rows Before: 114
Rows After: 21
Blank rows Removed: 93


,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1.0,Catcher in the Rye,"""20/02/2023""",25/02/2023,2 weeks,1.0
1,2.0,Lord of the rings the two towers,"""24/03/2023""",21/03/2023,2 weeks,2.0
2,3.0,Lord of the rings the return of the kind,"""29/03/2023""",25/03/2023,2 weeks,3.0
3,4.0,The hobbit,"""02/04/2023""",25/03/2023,2 weeks,4.0
4,5.0,Dune,"""02/04/2023""",25/03/2023,2 weeks,5.0
5,6.0,Little Women,"""02/04/2023""",01/05/2023,2 weeks,1.0
6,7.0,IT,"""10/04/2063""",03/04/2023,2 weeks,6.0
7,8.0,Misery,"""15/04/2023""",03/04/2023,2 weeks,7.0
8,9.0,Catch 22,"""15/04/2023""",16/04/2023,2 weeks,7.0
9,10.0,Animal Farm,"""20/04/2023""",24/04/2023,2 weeks,2.0


In [34]:
# Clean obvious text issues first
books_clean["Books"] = books_clean["Books"].astype("string").str.strip()

books_clean["Book checkout"] = (
    books_clean["Book checkout"]
    .astype("string")
    .str.replace('"', '', regex=False)
    .str.strip()
)

books_clean["Book Returned"] = (
    books_clean["Book Returned"]
    .astype("string")
    .str.replace('"', '', regex=False)
    .str.strip()
)

books_clean["Days allowed to borrow"] = (
    books_clean["Days allowed to borrow"]
    .astype("string")
    .str.strip()
)

# Set ID columns as whole numbers
books_clean["Id"] = pd.to_numeric(
    books_clean["Id"],
    errors="coerce"
).astype("Int64")

books_clean["Customer ID"] = pd.to_numeric(
    books_clean["Customer ID"],
    errors="coerce"
).astype("Int64")

# Set date columns as dates
books_clean["Book checkout"] = pd.to_datetime(
    books_clean["Book checkout"],
    errors="coerce",
    dayfirst=True
)

books_clean["Book Returned"] = pd.to_datetime(
    books_clean["Book Returned"],
    errors="coerce",
    dayfirst=True
)

# Check result
print(books_clean.dtypes)
display(books_clean)



Id                                 Int64
Books                     string[python]
Book checkout             datetime64[ns]
Book Returned             datetime64[ns]
Days allowed to borrow    string[python]
Customer ID                        Int64
dtype: object


,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1,Catcher in the Rye,2023-02-20,2023-02-25,2 weeks,1
1,2,Lord of the rings the two towers,2023-03-24,2023-03-21,2 weeks,2
2,3,Lord of the rings the return of the kind,2023-03-29,2023-03-25,2 weeks,3
3,4,The hobbit,2023-04-02,2023-03-25,2 weeks,4
4,5,Dune,2023-04-02,2023-03-25,2 weeks,5
5,6,Little Women,2023-04-02,2023-05-01,2 weeks,1
6,7,IT,2063-04-10,2023-04-03,2 weeks,6
7,8,Misery,2023-04-15,2023-04-03,2 weeks,7
8,9,Catch 22,2023-04-15,2023-04-16,2 weeks,7
9,10,Animal Farm,2023-04-20,2023-04-24,2 weeks,2


In [35]:
# Create a working copy
books_checked = books_clean.copy()

# Date columns to validate
date_columns = ["Book checkout", "Book Returned"]

# Rule 1: any column contains null
null_mask = books_checked.isna().any(axis=1)

# Rule 2: dates are missing, before 2023, or after 2026 
invalid_date_mask = pd.Series(False, index=books_checked.index)

for col in date_columns:
    invalid_date_mask = (
        invalid_date_mask
        | books_checked[col].isna()
        | (books_checked[col].dt.year < 2023)
        | (books_checked[col].dt.year > 2026)
    )

# Combined invalid rule
invalid_mask = null_mask | invalid_date_mask

# Split into valid and invalid dataframes 
books_invalid = books_checked[invalid_mask].copy()
books_valid = books_checked[~invalid_mask].copy()

print("Total rows:", len(books_checked)) 
print("Valid rows:", len(books_valid)) 
print("Invalid rows:", len(books_invalid))

# display(books_valid)
# display(books_invalid)

books_valid.to_csv("data/books_valid.csv")
books_invalid.to_csv("Data/books_invalid.csv ")

Total rows: 21
Valid rows: 18
Invalid rows: 3


In [36]:
customers_raw = customers.copy()

rows_before = len(customers_raw)

customers_clean = customers_raw.dropna(how="all").copy()

rows_after = len(customers_clean)

print("Rows Before:", rows_before)
print("Rows After:", rows_after)
print("Blank rows Removed:", rows_before - rows_after)

display(customers_clean)

Rows Before: 9
Rows After: 8
Blank rows Removed: 1


,Customer ID,Customer Name
0,1.0,Jane Doe
1,2.0,John Smith
2,3.0,Dan Reeves
4,5.0,William Holden
5,6.0,Jaztyn Forest
6,7.0,Jackie Irving
7,8.0,Matthew Stirling
8,9.0,Emory Ted


In [37]:
customers_clean["Customer ID"] = pd.to_numeric(
    customers_clean["Customer ID"],
    errors="coerce"
).astype("Int64")

customers_clean["Customer Name"] = (
    customers_clean["Customer Name"]
    .astype("string")
    .str.strip()
)

print(customers_clean.dtypes)
# display(customers_clean)
customers_valid = customers_clean
customers_valid.to_csv("data/customers_valid.csv")

Customer ID               Int64
Customer Name    string[python]
dtype: object
